In [1]:
print("OK")

OK


In [2]:
%pwd

'c:\\Users\\midem\\Documents\\medical-chatbot-with-langchain\\research'

In [3]:
import os
os.chdir('../')
%pwd

'c:\\Users\\midem\\Documents\\medical-chatbot-with-langchain'

In [4]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\midem\AppData\Local\Temp\ipykernel_9124\3754768683.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
c:\Users\midem\anaconda3\envs\medchatbot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Extract text from PDF files
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents

In [6]:
extracted_data = load_pdf_files("data")

In [7]:
extracted_data[0:10]

[Document(metadata={'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:00:02-05:00', 'moddate': '2004-12-18T16:15:31-06:00', 'source': 'data\\Medical_book.pdf', 'total_pages': 637, 'page': 0, 'page_label': '1'}, page_content=''),
 Document(metadata={'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:00:02-05:00', 'moddate': '2004-12-18T16:15:31-06:00', 'source': 'data\\Medical_book.pdf', 'total_pages': 637, 'page': 1, 'page_label': '2'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION'),
 Document(metadata={'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:00:02-05:00', 'moddate': '2004-12-18T16:15:31-06:00', 'source': 'data\\Medical_book.pdf', 'total_pages': 637, 'page': 2, 'page_label': '3'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION\nJACQUELINE L. LONGE, EDITOR\nDEIRDRE S. BLANCHFIELD, ASSOCIATE EDITOR\nVOLUME\nA-B\n1'),
 

In [8]:
len(extracted_data)

637

In [9]:
from typing import List
from langchain_core.documents import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs = []

    for doc in docs:

        text = doc.page_content.strip()

        if not text:
            continue

        minimal_docs.append(
            Document(
                page_content=text,
                metadata={
                    "source": doc.metadata.get("source"),
                    "page": doc.metadata.get("page"),
                },
            )
        )

    return minimal_docs

In [10]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [11]:
minimal_docs[0:20]

[Document(metadata={'source': 'data\\Medical_book.pdf', 'page': 1}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION'),
 Document(metadata={'source': 'data\\Medical_book.pdf', 'page': 2}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION\nJACQUELINE L. LONGE, EDITOR\nDEIRDRE S. BLANCHFIELD, ASSOCIATE EDITOR\nVOLUME\nA-B\n1'),
 Document(metadata={'source': 'data\\Medical_book.pdf', 'page': 3}, page_content='STAFF\nJacqueline L. Longe, Project Editor\nDeirdre S. Blanchfield, Associate Editor\nChristine B. Jeryan, Managing Editor\nDonna Olendorf, Senior Editor\nStacey Blachford, Associate Editor\nKate Kretschmann, Melissa C. McDade, Ryan\nThomason, Assistant Editors\nMark Springer, Technical Specialist\nAndrea Lopeman, Programmer/Analyst\nBarbara J. Yarrow,Manager, Imaging and Multimedia\nContent\nRobyn V . Young,Project Manager, Imaging and\nMultimedia Content\nDean Dauphinais, Senior Editor, Imaging and\nMultimedia Content\nKelly A. Quin, Editor, Imaging

In [12]:
# Split the documents into smaller chunks
def text_split(documents):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=100,
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    return splitter.split_documents(documents)

In [13]:
texts_chunk = text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunk)}")

Number of chunks: 3941


In [14]:
texts_chunk[0:20]

[Document(metadata={'source': 'data\\Medical_book.pdf', 'page': 1}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION'),
 Document(metadata={'source': 'data\\Medical_book.pdf', 'page': 2}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION\nJACQUELINE L. LONGE, EDITOR\nDEIRDRE S. BLANCHFIELD, ASSOCIATE EDITOR\nVOLUME\nA-B\n1'),
 Document(metadata={'source': 'data\\Medical_book.pdf', 'page': 3}, page_content='STAFF\nJacqueline L. Longe, Project Editor\nDeirdre S. Blanchfield, Associate Editor\nChristine B. Jeryan, Managing Editor\nDonna Olendorf, Senior Editor\nStacey Blachford, Associate Editor\nKate Kretschmann, Melissa C. McDade, Ryan\nThomason, Assistant Editors\nMark Springer, Technical Specialist\nAndrea Lopeman, Programmer/Analyst\nBarbara J. Yarrow,Manager, Imaging and Multimedia\nContent\nRobyn V . Young,Project Manager, Imaging and\nMultimedia Content\nDean Dauphinais, Senior Editor, Imaging and\nMultimedia Content\nKelly A. Quin, Editor, Imaging

In [15]:
from langchain_huggingface import HuggingFaceEmbeddings

def load_embeddings():
    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        encode_kwargs={"normalize_embeddings": True}
    )

embeddings = load_embeddings()

In [16]:
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={'normalize_embeddings': True}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [17]:
vector = embeddings.embed_query("Hello world")
vector

[-0.03447723388671875,
 0.031023165211081505,
 0.006734973751008511,
 0.026108991354703903,
 -0.039362046867609024,
 -0.16030244529247284,
 0.06692394614219666,
 -0.0064415112137794495,
 -0.04745052382349968,
 0.014758817851543427,
 0.07087530195713043,
 0.05552754923701286,
 0.019193369895219803,
 -0.026251282542943954,
 -0.01010951679199934,
 -0.02694047801196575,
 0.022307446226477623,
 -0.022226616740226746,
 -0.1496925950050354,
 -0.017493078485131264,
 0.007676276378333569,
 0.054352231323719025,
 0.0032544422429054976,
 0.03172587975859642,
 -0.0846213772892952,
 -0.029405953362584114,
 0.05159562826156616,
 0.04812408611178398,
 -0.0033148485235869884,
 -0.05827920883893967,
 0.04196930304169655,
 0.022210706025362015,
 0.128188818693161,
 -0.02233896777033806,
 -0.011656241491436958,
 0.06292831897735596,
 -0.032876331359148026,
 -0.09122606366872787,
 -0.031175393611192703,
 0.05269952118396759,
 0.047034814953804016,
 -0.0842030942440033,
 -0.030056195333600044,
 -0.02074482

In [18]:
print( "Vector length:", len(vector))

Vector length: 384


In [19]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [20]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY is not configured.")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is not configured.")

In [21]:
from pinecone import Pinecone 

pc = Pinecone(api_key=PINECONE_API_KEY)

In [22]:
pc

In [23]:
print(pc.list_indexes())

[{
    "name": "medical-chatbot",
    "metric": "cosine",
    "host": "medical-chatbot-52koa0o.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}, {
    "name": "insurance-claims-1",
    "metric": "euclidean",
    "host": "insurance-claims-1-52koa0o.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1536,
    "deletion_protection": "disabled",
    "tags": null
}, {
    "name": "insurance-claims",
    "metric": "euclidean",
    "host": "insurance-claims-52koa0o.svc.aped-4627-b74a.pinecone.io",
    "spec

In [24]:
index_name = "medical-chatbot"

if pc.has_index(index_name):
    pc.delete_index(index_name)

In [25]:
from pinecone import ServerlessSpec 

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384,  # Dimension of the embeddings
        metric= "cosine",  # Cosine similarity
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )


index = pc.Index(index_name)

In [26]:
from langchain_pinecone import PineconeVectorStore

namespace = "medical-encyclopedia"
docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embeddings,
    index_name=index_name,
    namespace=namespace
)

In [27]:
# Load Existing index 

from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings,
    namespace=namespace
)

### Add more data to the edxisting pinecone index

In [ ]:
#olumide = Document(
    #page_content="Olumide is Data Scientist, Machine Learning Engineer and AI enthusiast with background in process systems engineering, cloud and devops. ",
    #metadata={"source": "LinkedIn"}
#)

In [ ]:
#docsearch.add_documents(documents=[olumide])

['f4491596-545f-4745-b95b-7b19d2aa1ed2']

In [30]:
retriever = docsearch.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 10
    }
)

In [ ]:
#retrieved_docs = retriever.invoke("who is olumide")
#retrieved_docs

[Document(id='f4491596-545f-4745-b95b-7b19d2aa1ed2', metadata={'source': 'LinkedIn'}, page_content='Olumide is Data Scientist, Machine Learning Engineer and AI enthusiast with background in process systems engineering, cloud and devops. '),
 Document(id='f598ff8f-61c4-46f9-9ca7-18f11d2e9e3a', metadata={'source': 'data\\Medical_book.pdf'}, page_content='available (e.g., tumbling E, pictures, or letters). In ambly-\nopia, single letters are easier to recognize than when a\nKEY TERMS\nAnisometropia—An eye condition in which there\nis an inequality of vision between the two eyes.\nThere may be unequal amounts of nearsighted-\nness, farsightedness, or astigmatism, so that one\neye will be in focus while the other will not.\nCataract—Cloudiness of the eye’s natural lens.\nOcculsion therapy—A type of treatment for ambly-'),
 Document(id='64ab0a07-d03e-44dd-b59c-8e005f0f0753', metadata={'source': 'data\\Medical_book.pdf'}, page_content='available (e.g., tumbling E, pictures, or letters). In am

In [31]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(
    model="gpt-4o",
    temperature=0
)

In [33]:
#from langchain.chains import create_retrieval_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
#from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [34]:
system_prompt = """
You are a medical information assistant.

Answer the user's question using only the retrieved context.

Rules:
- Do not introduce facts that are not supported by the context.
- If there is insufficient information, say:
  "The available medical information does not provide enough
  information to answer this question."
- If sources conflict or are ambiguous, clearly state this.
- Explain technical medical terminology in plain language when useful.
- Do not provide a personalized diagnosis.
- Keep answers concise and factual.
- Where metadata is available, indicate the supporting source/page.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [35]:
question_answer_chain = create_stuff_documents_chain(chat_model, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [36]:
response = rag_chain.invoke({"input": "What is the cause of limited blood flow to the feet?"})
print(response["answer"])

The available medical information does not provide enough information to answer this question.


In [37]:
response = rag_chain.invoke({"input": "what is Acne?"})
print(response["answer"])

Acne is a skin condition that occurs when pores or hair follicles become blocked. This blockage allows a waxy material called sebum to collect inside the pores or follicles. Normally, sebum flows out onto the skin and hair to form a protective coating, but when it cannot get out, small swellings develop on the skin surface. Bacteria and dead skin cells can also collect, causing inflammation. Small, non-inflamed swellings are known as whiteheads or blackheads. When they become inflamed, they turn into pimples, and if they fill with pus, they are called pustules. Acne vulgaris, the medical term for common acne, is the most common skin disease, affecting nearly 17 million people in the United States. It usually begins at puberty and worsens during adolescence.


In [39]:
response = rag_chain.invoke({"input": "Hi"})
print(response["answer"])

Hello! How can I assist you with medical information today?
